# Criação das Dimensões — DW de Assistência Estudantil (UFPB, Campus I)

Este notebook cria e ajusta as **dimensões** utilizadas na tabela fato de
assistência estudantil, a partir dos dados do **SEDAP+ (Censo da Educação
Superior 2024)**.

**Dimensões geradas neste notebook:**
1. `dim_curso` (filtrada para o Campus I)
2. `dim_sexo`
3. `dim_raca`
4. `dim_turno` (com o registro adicional `"Não informado"`)
5. `dim_grau` (derivada de `dim_curso`)
6. `dim_modalidade` (derivada de `dim_curso`)

> As demais dimensões já prontas (`dim_centro`, `dim_auxilio`, etc.) **não**
> são recriadas aqui — este notebook trata apenas das dimensões que exigem
> algum tratamento (filtro de campus, mapeamento de descrição ou inclusão de
> registro "Não informado").

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd

## 2. Dimensão Curso (filtro Campus I)

O SEDAP+ não permite filtrar o Campus I diretamente na consulta SQL.
Por isso, o filtro é feito aqui em Python: removemos da `dim_curso` os
códigos de curso (`ID_CURSO`) que **não** pertencem ao Campus I.

Essa mesma lista de códigos (`cursos_fora`) será reutilizada no notebook da
tabela fato, para manter a consistência entre dimensão e fato.

In [ ]:
# Leitura da dimensão curso original (sem filtro de campus)
dim_curso = pd.read_csv("/content/dim_curso.csv")

In [ ]:
# Códigos de curso que NÃO pertencem ao Campus I
cursos_fora = [
    13403, 13454, 13455, 13457, 80589, 97767, 98976, 98980,
    98982, 98984, 99045, 107348, 107352, 107356, 107360,
    109626, 113709, 397767, 1161324, 1167933, 1440696,
    5000897, 5000898, 113699, 1110415, 113701
]

# Mantém apenas os cursos do Campus I
dim_curso = dim_curso[
    ~dim_curso["ID_CURSO"].isin(cursos_fora)
]

In [ ]:
print(f"Número de cursos (Campus I): {len(dim_curso)}")
display(dim_curso.head())

Número de cursos (Campus I): 93


,ID_CURSO,CURSO,CO_CINE_ROTULO,GRAU_ACADEMICO,MODALIDADE_ENSINO,TOTAL_ALUNOS
0,13397,CIÊNCIAS CONTÁBEIS,0411C01,Bacharelado,Presencial,1438
1,13418,PEDAGOGIA,0113P01,Licenciatura,Presencial,1382
2,13398,DIREITO,0421D01,Bacharelado,Presencial,1137
3,13424,MEDICINA,0912M01,Bacharelado,Presencial,844
4,107548,LETRAS - LÍNGUA PORTU,0115L13,Licenciatura,Presencial,766


In [ ]:
dim_curso.to_csv(
    "/content/dim_curso_campus1.csv",
    index=False,
    encoding="utf-8-sig"
)

## 3. Dimensão Sexo

In [ ]:
dim_sexo = pd.read_csv("/content/sexo_sedap.csv")

# Mapeamento de código -> descrição
sexo = {
    1: "Masculino",
    2: "Feminino"
}

dim_sexo["DESCRICAO"] = dim_sexo["TP_SEXO"].map(sexo)

dim_sexo = dim_sexo.rename(columns={
    "TP_SEXO": "ID_SEXO"
})

display(dim_sexo)

dim_sexo.to_csv(
    "/content/dim_sexo.csv",
    index=False,
    encoding="utf-8-sig"
)

,ID_SEXO,TOTAL_ALUNOS,DESCRICAO
0,1,21244,Masculino
1,2,18906,Feminino


## 4. Dimensão Raça

In [ ]:
dim_raca = pd.read_csv("/content/raca_sedap.csv")

# Mapeamento de código -> descrição
raca = {
    0: "Não declarada",
    1: "Branca",
    2: "Preta",
    3: "Parda",
    4: "Amarela",
    5: "Indígena"
}

dim_raca["DESCRICAO"] = dim_raca["TP_COR_RACA"].map(raca)

dim_raca = dim_raca.rename(columns={
    "TP_COR_RACA": "ID_RACA"
})

display(dim_raca)

dim_raca.to_csv(
    "/content/dim_raca.csv",
    index=False,
    encoding="utf-8-sig"
)

,ID_RACA,TOTAL_ALUNOS,DESCRICAO
0,0,3824,Não declarada
1,1,13113,Branca
2,2,3011,Preta
3,3,19256,Parda
4,4,384,Amarela
5,5,561,Indígena


## 5. Dimensão Turno (adição de "Não informado")

Existem cursos **EaD sem turno preenchido** no SEDAP+. Para não descartar
esses registros na fato, criamos aqui o registro `ID_TURNO = 0` ->
`"Não informado"`.

Na tabela fato, os valores nulos de turno serão substituídos por
`ID_TURNO = 0` (ver notebook da Fato, etapa "Atualizar a dimensão turno").

In [ ]:
dim_turno = pd.read_csv("/content/dim_turno.csv")

In [ ]:
# Novo registro para representar turno não informado (cursos EaD)
novo = pd.DataFrame({
    "ID_TURNO": [0],
    "TOTAL_ALUNOS": [0],
    "DESCRICAO": ["Não informado"]
})

dim_turno = pd.concat(
    [novo, dim_turno],
    ignore_index=True
)

dim_turno = (
    dim_turno
    .sort_values("ID_TURNO")
    .reset_index(drop=True)
)

display(dim_turno)

,ID_TURNO,TOTAL_ALUNOS,DESCRICAO
0,0,0,Não informado
1,1,4460,Matutino
2,2,3313,Vespertino
3,3,12663,Noturno
4,4,18479,Integral


In [ ]:
dim_turno.to_csv(
    "/content/dim_turno2.csv",
    index=False,
    encoding="utf-8-sig"
)

## 6. Dimensão Grau

Derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [ ]:
dim_grau = (
    dim_curso[["GRAU_ACADEMICO"]]
    .drop_duplicates()
    .sort_values("GRAU_ACADEMICO")
    .reset_index(drop=True)
)

dim_grau.insert(
    0,
    "ID_GRAU",
    range(1, len(dim_grau) + 1)
)

display(dim_grau)

dim_grau.to_csv(
    "/content/dim_grau.csv",
    index=False,
    encoding="utf-8-sig"
)

,ID_GRAU,GRAU_ACADEMICO
0,1,Bacharelado
1,2,Licenciatura
2,3,Tecnológico


## 7. Dimensão Modalidade

Também derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [ ]:
dim_modalidade = (
    dim_curso[["MODALIDADE_ENSINO"]]
    .drop_duplicates()
    .sort_values("MODALIDADE_ENSINO")
    .reset_index(drop=True)
)

dim_modalidade.insert(
    0,
    "ID_MODALIDADE",
    range(1, len(dim_modalidade) + 1)
)

display(dim_modalidade)

dim_modalidade.to_csv(
    "/content/dim_modalidade.csv",
    index=False,
    encoding="utf-8-sig"
)

,ID_MODALIDADE,MODALIDADE_ENSINO
0,1,EaD
1,2,Presencial
